# Dashboard interactivo - En proceso

In [3]:
import unicodedata
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import ipywidgets as ipw
from IPython.display import display, clear_output
import warnings

warnings.filterwarnings('ignore')

# Configuración y constantes

CSV_PATH = Path("Educación") / "casen_losrios_2017_2024.csv"
SHAPEFILE = Path("Vivienda") / "crawler" / "comunas" / "comunas.shp"
INDICADORES_CSV = Path("Vivienda") / "indicadores_vivienda.csv"
INDICADOR_MAPA = "carencia_servicios_basicos"
AÑOS_MAPA = [2017, 2024]

CODIGO_COMUNA = {
    '14101': 'Valdivia',
    '14102': 'Corral', 
    '14103': 'Lanco',
    '14104': 'Los Lagos',
    '14105': 'Máfil',
    '14106': 'Mariquina',
    '14107': 'Paillaco', 
    '14108': 'Panguipulli', 
    '14201': 'La Unión',
    '14202': 'Futrono',     
    '14203': 'Lago Ranco',
    '14204': 'Río Bueno',
}

NIVEL_LABEL = {
    1: 'Sin ed. formal',
    2: 'Básica',
    3: 'Básica',
    4: 'Básica',
    5: 'Básica',
    6: 'Media',
    7: 'Media',
    8: 'Media',
    9: 'Superior',
    10: 'Superior',
    11: 'Superior',
    12: 'Superior',
    13: 'Superior',
    14: 'Sin info',
    15: 'Sin info',
}
NIVELES_ORDEN = ['Sin ed. formal', 'Básica', 'Media', 'Superior']

C17 = '#185FA5'
C24 = '#EF9F27'
COK = '#1D9E75'
CWRN = '#E24B4A'
CBG = '#F9F9F7'

# Carga y procesamiento de datos

def cargar_datos(path):
    df = pd.read_csv(path)
    df['cod_comuna'] = df['estrato'].astype(str).str[:5]
    df['comuna'] = df['cod_comuna'].map(CODIGO_COMUNA)
    df['nivel_grupo'] = df['nivel_educacion'].map(NIVEL_LABEL)
    df['analfabeto'] = (df['alfabetismo'] != 1).astype(float)
    df['desertor'] = (
        df['edad'].between(6, 24) &
        (df['asiste_actualmente'] == 2) &
        df['nivel_educacion'].notna() &
        (df['nivel_educacion'] > 1)
    ).astype(float)
    return df

try:
    DF = cargar_datos(CSV_PATH)
except FileNotFoundError:
    raise FileNotFoundError(
        f"No se encontró '{CSV_PATH}'. Asegúrate de ejecutar este notebook desde la raíz del repo."
    )

COMUNAS_ALL = sorted(DF['comuna'].dropna().unique())
AÑOS = sorted(DF['año'].unique())


def normalizar_texto(valor):
    texto = unicodedata.normalize('NFKD', str(valor))
    texto = ''.join(c for c in texto if not unicodedata.combining(c))
    return ' '.join(texto.lower().split())

def cargar_mapas():
    gdf = gpd.read_file(SHAPEFILE)
    gdf = gdf[gdf['codregion'].astype(str) == '14'].copy()
    gdf['key'] = gdf['Comuna'].map(normalizar_texto)

    indicadores = pd.read_csv(INDICADORES_CSV, encoding='utf-8-sig')
    mapas = {}
    for año in AÑOS_MAPA:
        col = f'{INDICADOR_MAPA}_{año}'
        datos = indicadores[['comuna', col]].rename(columns={col: 'valor'})
        datos['key'] = datos['comuna'].map(normalizar_texto)
        mapas[año] = gdf.merge(datos, on='key', how='left')

    diferencia = mapas[AÑOS_MAPA[0]][['key', 'Comuna', 'geometry', 'valor']].rename(columns={'valor': 'valor_inicial'})
    diferencia = diferencia.merge(
        mapas[AÑOS_MAPA[1]][['key', 'valor']].rename(columns={'valor': 'valor_final'}),
        on='key',
    )
    diferencia['valor'] = diferencia['valor_final'] - diferencia['valor_inicial']
    return mapas, diferencia

try:
    MAPAS, DIFERENCIA_MAPA = cargar_mapas()
    LIMITE_DIFERENCIA = DIFERENCIA_MAPA['valor'].abs().max()
    MAPAS_OK = True
except Exception:
    MAPAS, DIFERENCIA_MAPA, LIMITE_DIFERENCIA, MAPAS_OK = {}, None, 0, False

# Cálculos métricas

def subconjunto(df, comunas, año):
    return df[df['comuna'].isin(comunas) & (df['año'] == año)]

def porcentaje(parte, base):
    return round(len(parte) / len(base) * 100, 1) if len(base) > 0 else 0

def sufijo_comunas(comunas):
    return f' ({", ".join(comunas)})' if len(comunas) <= 3 else f' ({len(comunas)} comunas)'


def tasa_desercion(df, comunas, año):
    sub = subconjunto(df, comunas, año)
    jovenes = sub[sub['edad'].between(6, 24)]
    return porcentaje(jovenes[jovenes['desertor'] == 1], jovenes)

def tasa_analfabetismo_jefes(df, comunas, año):
    sub = subconjunto(df, comunas, año)
    jefes = sub[sub['jefe_hogar'] == 1]
    return porcentaje(jefes[jefes['analfabeto'] == 1], jefes)

def tasa_exclusion_laboral(df, comunas, año):
    sub = subconjunto(df, comunas, año)
    jovenes = sub[sub['edad'].between(15, 29)]
    return porcentaje(jovenes[jovenes['desertor'] == 1], jovenes)

def tasa_sin_media_adultos(df, comunas, año):
    sub = subconjunto(df, comunas, año)
    adultos = sub[sub['edad'].between(15, 64)]
    return porcentaje(adultos[adultos['nivel_grupo'].isin(['Sin ed. formal', 'Básica'])], adultos)

def tasa_adultos_mayores(df, comunas, año):
    sub = subconjunto(df, comunas, año)
    return porcentaje(sub[sub['edad'] >= 65], sub)

def tasa_ninos(df, comunas, año):
    sub = subconjunto(df, comunas, año)
    return porcentaje(sub[sub['edad'] < 15], sub)


def kpi_educacion(df, comunas, año):
    sub = subconjunto(df, comunas, año)
    jefes = sub[sub['jefe_hogar'] == 1]
    kpi1 = porcentaje(jefes[jefes['nivel_grupo'].isin(['Sin ed. formal', 'Básica'])], jefes)
    return kpi1, tasa_desercion(df, comunas, año)

def vol_educacion(df, comunas, año):
    sub = subconjunto(df, comunas, año)
    return int((sub['jefe_hogar'] == 1).sum()), int(sub['edad'].between(6, 24).sum())

def tabla_niveles(df, comunas, año):
    sub = subconjunto(df, comunas, año)
    jefes = sub[sub['jefe_hogar'] == 1]
    counts = jefes['nivel_grupo'].value_counts()
    total = counts.sum()
    return {nv: round(counts.get(nv, 0) / total * 100, 1) for nv in NIVELES_ORDEN}


def aplicar_fA(personas, dormitorios):
    if dormitorios <= 0:
        return None, None, None
    A = round(personas / dormitorios, 2)
    if A < 2.5:
        return 1, 'Sin hacinamiento', A
    elif A < 3.5:
        return 2, 'Hacinamiento medio', A
    elif A < 5.0:
        return 3, 'Hacinamiento alto', A
    else:
        return 4, 'Hacinamiento crítico', A


def kpi_vivienda(df, comunas, año):
    if not MAPAS_OK:
        return 0, 0
    claves = [normalizar_texto(c) for c in comunas]
    mapa_año = MAPAS.get(año)
    carencia = mapa_año[mapa_año['key'].isin(claves)]['valor'] if mapa_año is not None else pd.Series(dtype=float)
    cambio = DIFERENCIA_MAPA[DIFERENCIA_MAPA['key'].isin(claves)]['valor']
    kpi1 = round(carencia.mean(), 1) if len(carencia) else 0
    kpi2 = round(cambio.mean(), 1) if len(cambio) else 0
    return kpi1, kpi2

def vol_vivienda(df, comunas, año):
    sub = subconjunto(df, comunas, año)
    hogares = sub['id_hogar'].nunique()
    personas_por_hogar = round(len(sub) / hogares, 1) if hogares else 0
    return hogares, personas_por_hogar


def kpi_empleo(df, comunas, año):
    return tasa_exclusion_laboral(df, comunas, año), tasa_sin_media_adultos(df, comunas, año)

def vol_empleo(df, comunas, año):
    sub = subconjunto(df, comunas, año)
    return int(sub['edad'].between(15, 64).sum()), int(len(sub))


def kpi_composicion(df, comunas, año):
    sub = subconjunto(df, comunas, año)
    jefes = sub[sub['jefe_hogar'] == 1]
    kpi1 = porcentaje(jefes[jefes['analfabeto'] == 1], jefes)
    return kpi1, tasa_adultos_mayores(df, comunas, año)

def vol_composicion(df, comunas, año):
    sub = subconjunto(df, comunas, año)
    return sub['id_hogar'].nunique(), int(len(sub))

# Funciones de graficado

def estilo_ax(ax, titulo='', xlabel='', ylabel=''):
    ax.set_facecolor('white')
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines[['left', 'bottom']].set_color('#DDDDDD')
    ax.tick_params(colors='#555', labelsize=9)
    if titulo:
        ax.set_title(titulo, fontsize=10, color='#222', pad=8, fontweight='normal')
    if xlabel:
        ax.set_xlabel(xlabel, fontsize=9, color='#666')
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=9, color='#666')


def plot_barras_educacion(ax, comunas, sufijo=''):
    x = np.arange(len(NIVELES_ORDEN))
    w = 0.35
    for j, (año, color) in enumerate([(2017, C17), (2024, C24)]):
        vals = [tabla_niveles(DF, comunas, año).get(n, 0) for n in NIVELES_ORDEN]
        bars = ax.bar(x + (j - 0.5) * w, vals, w, color=color, label=str(año), zorder=3, alpha=0.9)
        for bar in bars:
            h = bar.get_height()
            if h > 0:
                ax.text(bar.get_x() + bar.get_width() / 2, h + 0.5,
                        f'{h:.1f}%', ha='center', va='bottom', fontsize=7.5, color='#444')
    ax.set_xticks(x)
    ax.set_xticklabels(NIVELES_ORDEN, fontsize=9)
    ax.yaxis.grid(True, color='#EEEEEE', zorder=0)
    ax.set_axisbelow(True)
    ax.legend(fontsize=8, framealpha=0)
    estilo_ax(ax, titulo=f'Nivel educacional, jefes de hogar{sufijo}', ylabel='%')


def plot_hacinamiento(ax, personas, dormitorios):
    f, label, A = aplicar_fA(personas, dormitorios)
    if f is None:
        ax.text(0.5, 0.5, 'Dormitorios debe ser > 0', ha='center', va='center', transform=ax.transAxes)
        return None, None, None

    cats = ['Sin hacinamiento\nf = 1', 'Hacinamiento medio\nf = 2',
            'Hacinamiento alto\nf = 3', 'Hacinamiento crítico\nf = 4']
    rangos = ['< 2,5 p/dorm', '2,5 – 3,5', '3,5 – 5,0', '≥ 5,0']
    colores = [COK, C24, '#D85A30', CWRN]
    anchos = [2.5, 1.0, 1.5, 2.0]
    alphas = [1.0 if i + 1 == f else 0.22 for i in range(4)]

    bars = ax.barh(cats, anchos, color=colores, height=0.52, zorder=3)
    for bar, a in zip(bars, alphas):
        bar.set_alpha(a)
    ax.set_xlim(0, 8)
    for i, (bar, rango) in enumerate(zip(bars, rangos)):
        c = '#222' if alphas[i] == 1.0 else '#999'
        ax.text(bar.get_width() + 0.15, bar.get_y() + bar.get_height() / 2,
                rango, va='center', fontsize=8.5, color=c)

    ax.set_ylim(-0.9, 3.5)
    pos = min(A, 7.5)
    ax.axvline(pos, color='#222', lw=1.8, ls='--', zorder=5)
    ax.text(pos, -0.7, f'A = {A}', ha='center', va='center', fontsize=8.5, color='#222',
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#CCC', lw=0.8))

    ax.xaxis.set_visible(False)
    ax.spines[['top', 'right', 'bottom']].set_visible(False)
    ax.spines['left'].set_color('#DDD')
    estilo_ax(ax, titulo=f'Función de hacinamiento f(A): {personas} personas / {dormitorios} dorm.')
    return f, label, A


def plot_horizontal_comunas(ax, comunas, año, col_fn, titulo, xlabel):
    registros = sorted(((c, col_fn(DF, [c], año)) for c in comunas), key=lambda r: r[1], reverse=True)
    labels = [r[0] for r in registros]
    vals = [r[1] for r in registros]
    color = C17 if año == 2017 else C24

    bars = ax.barh(labels, vals, color=color, height=0.55, zorder=3, alpha=0.9)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height() / 2,
                f'{v:.1f}%', va='center', fontsize=8, color='#444')
    ax.xaxis.grid(True, color='#EEEEEE', zorder=0)
    ax.set_axisbelow(True)
    ax.set_xlim(0, max(vals or [0]) + 8)
    estilo_ax(ax, titulo=titulo, xlabel=xlabel)


def plot_mapa_carencia(ax, mapa, comunas, cmap, escala, titulo, etiqueta_barra):
    mapa.plot(column='valor', cmap=cmap, edgecolor='black', linewidth=0.5, ax=ax,
              legend=True, legend_kwds={'label': etiqueta_barra, 'shrink': 0.7},
              missing_kwds={'color': 'lightgrey'}, **escala)
    claves = [normalizar_texto(c) for c in comunas]
    seleccion = mapa[mapa['key'].isin(claves)]
    if not seleccion.empty:
        seleccion.boundary.plot(ax=ax, edgecolor=COK, linewidth=2.2, zorder=4)
    ax.set_facecolor(CBG)
    ax.set_title(titulo, fontsize=10, color='#222')
    ax.set_axis_off()

# Interfaz y componentes reutilizables

def html_kpi(etiqueta, valor, unidad, color_borde):
    return f"""
    <div style="background:#F4F6FA;border-radius:8px;padding:9px 14px;
                border-left:4px solid {color_borde};min-width:168px">
      <div style="font-size:11px;color:#777;margin-bottom:3px">{etiqueta}</div>
      <div style="font-size:20px;font-weight:500;color:#1A1A1A">
        {valor}<span style="font-size:12px;color:#999;margin-left:2px">{unidad}</span>
      </div>
    </div>"""

def html_vol(etiqueta, valor):
    return f"""
    <div style="background:#F0F0EE;border-radius:8px;padding:9px 14px;min-width:140px">
      <div style="font-size:11px;color:#888;margin-bottom:3px">{etiqueta}</div>
      <div style="font-size:18px;font-weight:500;color:#444">{valor:,}</div>
    </div>"""


def crear_tab(cfg):
    dd_año = ipw.Dropdown(options=[(str(a), a) for a in AÑOS], value=AÑOS[0],
                          layout=ipw.Layout(width='90px'))
    dd_graf = ipw.Dropdown(options=cfg.get('graf_opciones', [('Vista principal', 'main')]),
                           layout=ipw.Layout(width='230px'))

    kpi1_w = ipw.HTML()
    kpi2_w = ipw.HTML()
    vol1_w = ipw.HTML()
    vol2_w = ipw.HTML()
    nota_w = ipw.HTML(f'<div style="font-size:10px;color:#bbb;margin-top:2px">Fuente: {cfg["fuente"]}</div>')
    out_fig = ipw.Output()
    out_result = ipw.HTML()

    def refrescar(comunas_sel, año_sel, graf_sel, extras_vals):
        v1, v2 = cfg['kpi_fn'](DF, comunas_sel, año_sel)
        n1, n2 = cfg['vol_fn'](DF, comunas_sel, año_sel)
        kpi1_w.value = html_kpi(cfg['kpi_labels'][0], v1, cfg['kpi_unidades'][0], cfg['kpi_colores'][0])
        kpi2_w.value = html_kpi(cfg['kpi_labels'][1], v2, cfg['kpi_unidades'][1], cfg['kpi_colores'][1])
        vol1_w.value = html_vol(cfg['vol_labels'][0], n1)
        vol2_w.value = html_vol(cfg['vol_labels'][1], n2)
        
        # Corrección: Limpiar el HTML auxiliar en cada refresco de gráfico
        out_result.value = ""
        
        with out_fig:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(7.2, 3.8))
            fig.patch.set_facecolor(CBG)
            ax.set_facecolor('white')
            resultado = cfg['render_fn'](ax, comunas_sel, año_sel, graf_sel, extras_vals)
            plt.tight_layout(pad=1.2)
            plt.show()
            if resultado is not None:
                out_result.value = resultado

    encabezado = ipw.HTML(f"""
    <div style="padding:6px 0 8px;border-bottom:1.5px solid #E8E8E8;margin-bottom:10px">
      <span style="font-size:15px;font-weight:500;color:#1A1A1A">{cfg['nombre']}</span>
    </div>""")
    lbl_kpis = ipw.HTML('<div style="font-size:11px;color:#999;margin-bottom:5px">KPIs del estudio</div>')
    lbl_vol = ipw.HTML('<div style="font-size:11px;color:#999;margin-bottom:5px">Volumen de muestra</div>')
    fila_kpi = ipw.HBox(
        [ipw.VBox([lbl_kpis, ipw.HBox([kpi1_w, kpi2_w], layout=ipw.Layout(gap='8px'))]),
         ipw.VBox([lbl_vol,  ipw.HBox([vol1_w, vol2_w], layout=ipw.Layout(gap='8px'))])],
        layout=ipw.Layout(gap='20px', align_items='flex-start', margin='0 0 12px 0')
    )
    bloque_año = ipw.HBox(
        [ipw.HTML('<span style="font-size:11px;color:#999;line-height:28px;margin-left:10px">Año:</span>'), dd_año],
        layout=ipw.Layout(align_items='center', gap='5px')
    )
    controles = ipw.HBox(
        [ipw.HTML('<span style="font-size:11px;color:#999;line-height:28px">Gráfico:</span>'), dd_graf, bloque_año],
        layout=ipw.Layout(align_items='center', gap='5px', margin='0 0 8px 0')
    )
    extras_box = ipw.VBox(cfg.get('extras_widgets', []), layout=ipw.Layout(margin='0 0 6px 0'))
    contenido = ipw.VBox([encabezado, fila_kpi, nota_w, controles, extras_box, out_fig, out_result],
                         layout=ipw.Layout(padding='14px'))

    graf_con_extras = cfg.get('graf_con_extras', set())
    graf_sin_año = cfg.get('graf_sin_año', set())

    def alternar_controles(*_):
        extras_box.layout.display = '' if dd_graf.value in graf_con_extras else 'none'
        bloque_año.layout.display = 'none' if dd_graf.value in graf_sin_año else ''

    dd_graf.observe(alternar_controles, names='value')
    alternar_controles()

    return contenido, refrescar, dd_año, dd_graf

# Educación

def render_educacion(ax, comunas, año, graf, extras):
    sufijo = sufijo_comunas(comunas)
    if graf == 'barras':
        plot_barras_educacion(ax, comunas, sufijo)
        return
    opciones = {
        'desercion': (tasa_desercion, f'Tasa de deserción escolar 6–24 años{sufijo}  ({año})', '%'),
        'analfabetismo': (tasa_analfabetismo_jefes, f'Analfabetismo en jefes de hogar{sufijo}  ({año})', '%'),
    }
    fn, titulo, xlabel = opciones[graf]
    plot_horizontal_comunas(ax, comunas, año, col_fn=fn, titulo=titulo, xlabel=xlabel)

cfg_edu = dict(
    nombre='Educación',
    kpi_fn=kpi_educacion,
    kpi_labels=['Jefes sin media completa', 'Deserción escolar 6–24 años'],
    kpi_unidades=['%', '%'],
    kpi_colores=[C17, CWRN],
    vol_fn=vol_educacion,
    vol_labels=['Jefes de hogar', 'Jóvenes 6–24 años'],
    graf_opciones=[
        ('Barras por nivel educacional', 'barras'),
        ('Deserción escolar por comuna', 'desercion'),
        ('Analfabetismo jefes por comuna', 'analfabetismo'),
    ],
    render_fn=render_educacion,
    extras_widgets=[],
    fuente='CASEN 2017 y 2024, Región de Los Ríos'
)

# Vivienda

sl_personas = ipw.IntSlider(min=1, max=15, value=4, step=1, description='Personas:',
                            continuous_update=False, style={'description_width': '75px'},
                            layout=ipw.Layout(width='310px'))
sl_dorm = ipw.IntSlider(min=1, max=8, value=2, step=1, description='Dorm.:',
                        continuous_update=False, style={'description_width': '75px'},
                        layout=ipw.Layout(width='270px'))

COLORES_FA = {1: COK, 2: C24, 3: '#D85A30', 4: CWRN}

def resultado_hacinamiento(f, label, A):
    if f is None:
        return None
    c = COLORES_FA[f]
    return (f'<div style="margin-top:5px;padding:8px 14px;background:#F8F8F6;'
            f'border-radius:8px;border-left:4px solid {c}">'
            f'<span style="font-size:13px;color:#444">'
            f'Índice A = <b>{A}</b> personas/dormitorio, '
            f'f(A) = <b style="color:{c}">{f} ({label})</b></span></div>')

def render_vivienda(ax, comunas, año, graf, extras):
    if graf == 'fA':
        personas, dormitorios = extras
        f, label, A = plot_hacinamiento(ax, personas, dormitorios)
        return resultado_hacinamiento(f, label, A)

    if not MAPAS_OK:
        ax.text(0.5, 0.5, 'No se encontró el shapefile de comunas o indicadores_vivienda.csv\n'
                          '(deben estar en Vivienda/crawler/comunas/ y Vivienda/ del repo)',
                ha='center', va='center', transform=ax.transAxes, fontsize=9, color='#999')
        ax.set_axis_off()
        return None

    if graf == 'mapa_carencia':
        plot_mapa_carencia(ax, MAPAS[año], comunas, 'YlOrRd', {},
                           f'Carencia de servicios básicos por comuna ({año})', 'Porcentaje (%)')
    elif graf == 'mapa_diferencia':
        plot_mapa_carencia(ax, DIFERENCIA_MAPA, comunas, 'RdBu_r',
                           {'vmin': -LIMITE_DIFERENCIA, 'vmax': LIMITE_DIFERENCIA},
                           f'Cambio en la carencia de servicios básicos, {AÑOS_MAPA[0]} a {AÑOS_MAPA[1]}',
                           'Cambio en puntos porcentuales')

cfg_viv = dict(
    nombre='Vivienda y Habitabilidad',
    kpi_fn=kpi_vivienda,
    kpi_labels=['Carencia de servicios básicos', f'Cambio {AÑOS_MAPA[0]} a {AÑOS_MAPA[1]}'],
    kpi_unidades=['%', 'p.p.'],
    kpi_colores=[CWRN, C17],
    vol_fn=vol_vivienda,
    vol_labels=['Hogares únicos', 'Personas por hogar'],
    graf_opciones=[
        ('Simulador de hacinamiento f(A)', 'fA'),
        ('Mapa de carencia de servicios básicos', 'mapa_carencia'),
        ('Mapa de cambio entre 2017 y 2024', 'mapa_diferencia'),
    ],
    graf_con_extras={'fA'},
    graf_sin_año={'mapa_diferencia'},
    render_fn=render_vivienda,
    extras_widgets=[sl_personas, sl_dorm],
    fuente='CASEN 2017 y 2024 (hogares) / BCN (carencia de servicios básicos por comuna)'
)

# Empleo e Ingresos

def render_empleo(ax, comunas, año, graf, extras):
    sufijo = sufijo_comunas(comunas)
    opciones = {
        'excl_laboral': (tasa_exclusion_laboral,
                         f'Exclusión laboral temprana 15–29 años{sufijo}  ({año})',
                         '% jóvenes sin asistencia ni formación (proxy)'),
        'edu_adultos': (tasa_sin_media_adultos,
                        f'Adultos 15–64 sin educación media completa{sufijo}  ({año})', '%'),
    }
    fn, titulo, xlabel = opciones[graf]
    plot_horizontal_comunas(ax, comunas, año, col_fn=fn, titulo=titulo, xlabel=xlabel)

cfg_emp = dict(
    nombre='Empleo e Ingresos',
    kpi_fn=kpi_empleo,
    kpi_labels=['Excl. laboral 15–29 (proxy)', 'Sin media en edad laboral (proxy)'],
    kpi_unidades=['%', '%'],
    kpi_colores=[C17, '#D85A30'],
    vol_fn=vol_empleo,
    vol_labels=['Adultos 15–64', 'Total personas'],
    graf_opciones=[
        ('Exclusión laboral temprana por comuna', 'excl_laboral'),
        ('Adultos sin ed. media por comuna', 'edu_adultos'),
    ],
    render_fn=render_empleo,
    extras_widgets=[],
    fuente='CASEN 2017 y 2024 / ENE INE (proxies sobre columnas disponibles)'
)

# Composición del Hogar

def render_composicion(ax, comunas, año, graf, extras):
    sufijo = sufijo_comunas(comunas)
    opciones = {
        'analfabetismo': (tasa_analfabetismo_jefes,
                          f'Analfabetismo jefes de hogar{sufijo}  ({año})', '% jefes analfabetos'),
        'adultos_mayores': (tasa_adultos_mayores,
                            f'Proporción adultos mayores ≥65{sufijo}  ({año})', '% sobre total de personas'),
        'ninos': (tasa_ninos,
                  f'Proporción de niños <15 años{sufijo}  ({año})', '% sobre total de personas'),
    }
    fn, titulo, xlabel = opciones[graf]
    plot_horizontal_comunas(ax, comunas, año, col_fn=fn, titulo=titulo, xlabel=xlabel)

cfg_comp = dict(
    nombre='Composición del Hogar',
    kpi_fn=kpi_composicion,
    kpi_labels=['Analfabetismo jefes de hogar', 'Adultos mayores ≥65 en muestra'],
    kpi_unidades=['%', '%'],
    kpi_colores=[CWRN, COK],
    vol_fn=vol_composicion,
    vol_labels=['Hogares únicos', 'Total personas'],
    graf_opciones=[
        ('Analfabetismo jefes por comuna', 'analfabetismo'),
        ('Proporción adultos mayores por comuna', 'adultos_mayores'),
        ('Proporción niños <15 años por comuna', 'ninos'),
    ],
    render_fn=render_composicion,
    extras_widgets=[],
    fuente='CASEN 2017 y 2024 / CENSO 2017 y 2024'
)

# Configuración del Dashboard

CONFIGS = [cfg_edu, cfg_viv, cfg_emp, cfg_comp]
tab_widgets, refreshers, dd_años_list, dd_grafs_list = [], [], [], []

for cfg in CONFIGS:
    cont, ref, dda, ddg = crear_tab(cfg)
    tab_widgets.append(cont)
    refreshers.append(ref)
    dd_años_list.append(dda)
    dd_grafs_list.append(ddg)

tabs = ipw.Tab(children=tab_widgets, layout=ipw.Layout(flex='1'))
for i, cfg in enumerate(CONFIGS):
    tabs.set_title(i, cfg['nombre'])

sel_comunas = ipw.SelectMultiple(options=COMUNAS_ALL, value=COMUNAS_ALL[:3], rows=12,
                                 layout=ipw.Layout(width='155px', height='260px'))
btn_ok = ipw.Button(description='Actualizar', layout=ipw.Layout(width='135px', margin='8px 0 0 0'))
panel_izq = ipw.VBox(
    [ipw.HTML('<div style="font-size:11px;color:#888;margin-bottom:4px">Comunas:</div>'), sel_comunas, btn_ok],
    layout=ipw.Layout(padding='14px 10px 14px 14px', min_width='175px', border_right='1.5px solid #EBEBEB')
)

titulo = ipw.HTML("""
<div style="background:#185FA5;color:white;padding:11px 18px;border-radius:8px 8px 0 0">
  <div style="font-size:15px;font-weight:500">Pobreza Multidimensional en la Región de Los Ríos</div>
  <div style="font-size:10px;opacity:0.75;margin-top:2px">Fuente: CASEN 2017 y 2024 · CENSO · ENE · BCN · Portal Inmobiliario</div>
</div>""")

cuerpo = ipw.HBox([panel_izq, tabs],
                  layout=ipw.Layout(border='1px solid #E0E0E0', border_radius='0 0 8px 8px',
                                    background='white', min_height='480px'))
dashboard = ipw.VBox([titulo, cuerpo], layout=ipw.Layout(max_width='980px'))

# Controladores de eventos

def _extras_vals(tab_i):
    return (sl_personas.value, sl_dorm.value) if tab_i == 1 else ()

def _refrescar_tab(i):
    comunas = list(sel_comunas.value) or COMUNAS_ALL
    año, graf, extras = dd_años_list[i].value, dd_grafs_list[i].value, _extras_vals(i)
    refreshers[i](comunas, año, graf, extras)

def _refrescar_actual(*_):
    _refrescar_tab(tabs.selected_index)

btn_ok.on_click(_refrescar_actual)
tabs.observe(lambda *_: _refrescar_tab(tabs.selected_index), names='selected_index')

for i in range(len(CONFIGS)):
    dd_años_list[i].observe(lambda c, idx=i: _refrescar_tab(idx), names='value')
    dd_grafs_list[i].observe(lambda c, idx=i: _refrescar_tab(idx), names='value')

sl_personas.observe(lambda c: _refrescar_tab(1), names='value')
sl_dorm.observe(lambda c: _refrescar_tab(1), names='value')

# Ejecución

display(dashboard)
_refrescar_tab(0)